In [1]:
# ==========================================
# 11주차: 데이터 로드 및 회귀분석 대상 설정 (code/11_regression.ipynb)
# ==========================================
!pip install pyreadstat statsmodels

from google.colab import drive
import numpy as np
import pandas as pd
import pyreadstat
import statsmodels.api as sm
import statsmodels.formula.api as smf

drive.mount('/content/drive')

# 1. 데이터 로드 (10주차와 동일)
DATA_DIR = '/content/drive/MyDrive/hdac_data'
target_cols = [
    'age', 'sex', 'kstrata', 'psu', 'wt_ntr',
    'genertn', 'HEI', 'incm', 'edu', 'BD1_11', 'pa_aerobic'
]

df, meta = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', usecols=target_cols)
if 'HEI' in df.columns:
    df['HE_khei'] = df['HEI']

# 2. 10주차 전처리 동일 적용
df['genertn_rc'] = df['genertn'].apply(lambda x: 1.0 if x == 1.0 else (0.0 if (2.0 <= x <= 7.0) else np.nan))
df['BD1_11_rc'] = df['BD1_11'].replace(9.0, np.nan) # 무응답 결측 처리

# 10주차 기본 분석 대상 (subpop_t1)
cond_adult = df['age'] >= 20
cond_wt = df['wt_ntr'].notnull()
cond_exp = df['genertn_rc'].notnull()
cond_out = df['HE_khei'].notnull()
df['subpop_t1'] = np.where(cond_adult & cond_wt & cond_exp & cond_out, 1.0, 0.0)

# 3. 11주차 회귀분석 보정변수 완전 사례(Complete Case) 확인
# 최소 보정집합: age, sex, incm, edu
covars = ['age', 'sex', 'incm', 'edu']
cond_covars = df[covars].notnull().all(axis=1)

# 회귀분석 최종 분석 대상 (subpop_reg)
df['subpop_reg'] = np.where((df['subpop_t1'] == 1.0) & cond_covars, 1.0, 0.0)

print('=' * 50)
print(f"1. Table 1 분석 대상자 (subpop_t1 == 1): {int(df['subpop_t1'].sum()):,}명")
print(f"2. 회귀분석 완전사례 대상자 (subpop_reg == 1): {int(df['subpop_reg'].sum()):,}명")
print(f"3. 보정변수 결측으로 인한 제외 인원: {int(df['subpop_t1'].sum() - df['subpop_reg'].sum()):,}명")
print('=' * 50)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 14.2 MB/s  0:00:00
Mounted at /content/drive
1. Table 1 분석 대상자 (subpop_t1 == 1): 5,816명
2. 회귀분석 완전사례 대상자 (subpop_reg == 1): 5,652명
3. 보정변수 결측으로 인한 제외 인원: 164명


In [2]:
# ==========================================
# 11주차: 복합표본 선형회귀분석 모형 1 및 모형 2 적합 (오류 보정)
# ==========================================
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. 완전 사례 분석 대상자(subpop_reg == 1.0)만 명확히 필터링한 서브셋 데이터프레임 생성
# (결측치가 제거된 상태이므로 statsmodels 내부 길이 불일치 오류 예방)
df_reg = df[df['subpop_reg'] == 1.0].copy()

# 2. 모형 1 (Unadjusted): KHEI ~ 1인 가구 여부 (genertn_rc)
formula_m1 = "HE_khei ~ genertn_rc"
model_m1 = smf.wls(formula_m1, data=df_reg, weights=df_reg['wt_ntr'])
result_m1 = model_m1.fit(cov_type='cluster', cov_kwds={'groups': df_reg['psu']})

# 3. 모형 2 (Adjusted): KHEI ~ 1인 가구 여부 + 최소 보정집합(연령, 성별, 소득 분위, 교육수준)
# 범주형 변수는 C()로 가변수(Dummy Variable) 처리
formula_m2 = "HE_khei ~ genertn_rc + age + C(sex) + C(incm) + C(edu)"
model_m2 = smf.wls(formula_m2, data=df_reg, weights=df_reg['wt_ntr'])
result_m2 = model_m2.fit(cov_type='cluster', cov_kwds={'groups': df_reg['psu']})

# 4. Table 2 자동 출력 함수
def print_regression_results(result, model_name):
    print(f"\n================ {model_name} 결과 ================")
    params = result.params['genertn_rc']
    conf_int = result.conf_int().loc['genertn_rc']
    pval = result.pvalues['genertn_rc']
    se_val = result.bse['genertn_rc']
    nobs = len(df_reg)

    print(f"• 회귀계수 (Beta): {params:.2f}")
    print(f"• 표준오차 (SE): {se_val:.2f}")
    print(f"• 95% 신뢰구간 (CI): [{conf_int[0]:.2f}, {conf_int[1]:.2f}]")
    print(f"• p-value: {pval:.4f}")
    print(f"• 분석 인원 (n): {nobs:,}명")
    print("--------------------------------------------------")

print_regression_results(result_m1, "모형 1 (보정 전 Unadjusted)")
print_regression_results(result_m2, "모형 2 (최소 보정집합 보정 후 Adjusted)")


================ 모형 1 (보정 전 Unadjusted) 결과 ================
• 회귀계수 (Beta): -2.38
• 표준오차 (SE): 0.80
• 95% 신뢰구간 (CI): [-3.95, -0.82]
• p-value: 0.0029
• 분석 인원 (n): 5,652명
--------------------------------------------------

================ 모형 2 (최소 보정집합 보정 후 Adjusted) 결과 ================
• 회귀계수 (Beta): -2.49
• 표준오차 (SE): 0.47
• 95% 신뢰구간 (CI): [-3.41, -1.56]
• p-value: 0.0000
• 분석 인원 (n): 5,652명
--------------------------------------------------
